# Serving ML Models with FastAPI

## Why FastAPI for ML Serving?

| Feature | Flask | FastAPI |
|---------|-------|--------|
| Performance | Medium | High (async) |
| Type validation | Manual | Automatic (Pydantic) |
| Auto docs | ❌ | ✅ Swagger + ReDoc |
| Async support | Limited | Native |
| Batch inference | Manual | Easy |
| WebSockets | Plugin | Built-in |

## ML Serving Architecture

```
Client → Load Balancer → FastAPI Workers → Model (in memory)
                                        → Feature Store
                                        → Model Registry
                                        → Monitoring
```

In [1]:
# pip install fastapi uvicorn scikit-learn joblib numpy

# Step 1: Train and save a model
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import numpy as np

# Train
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train_scaled, y_train)

# Save
joblib.dump(model, "iris_model.pkl")
joblib.dump(scaler, "iris_scaler.pkl")
print(f"Model accuracy: {model.score(scaler.transform(X_test), y_test):.3f}")

Model accuracy: 0.967


In [2]:
# ml_api.py Complete sklearn serving app

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
from typing import List, Optional
import joblib
import numpy as np
import time

# --- Models ---
class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., ge=0, le=10)
    sepal_width: float = Field(..., ge=0, le=10)
    petal_length: float = Field(..., ge=0, le=10)
    petal_width: float = Field(..., ge=0, le=10)

class BatchFeatures(BaseModel):
    samples: List[IrisFeatures]

class Prediction(BaseModel):
    class_id: int
    class_name: str
    probability: float
    all_probabilities: List[float]
    latency_ms: float

CLASS_NAMES = ["setosa", "versicolor", "virginica"]
models = {}  # shared model registry

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Load at startup not on every request!
    models["classifier"] = joblib.load("iris_model.pkl")
    models["scaler"] = joblib.load("iris_scaler.pkl")
    print("Models loaded successfully")
    yield
    models.clear()

app = FastAPI(title="Iris Classifier API", lifespan=lifespan)

@app.get("/health")
async def health():
    return {"status": "healthy", "models_loaded": list(models.keys())}

@app.post("/predict", response_model=Prediction)
async def predict(features: IrisFeatures):
    start = time.time()
    X = np.array([[features.sepal_length, features.sepal_width,
                   features.petal_length, features.petal_width]])
    X_scaled = models["scaler"].transform(X)
    class_id = int(models["classifier"].predict(X_scaled)[0])
    probs = models["classifier"].predict_proba(X_scaled)[0].tolist()
    latency = (time.time() - start) * 1000
    return Prediction(
        class_id=class_id,
        class_name=CLASS_NAMES[class_id],
        probability=probs[class_id],
        all_probabilities=probs,
        latency_ms=round(latency, 2)
    )

@app.post("/predict/batch")
async def predict_batch(batch: BatchFeatures):
    start = time.time()
    X = np.array([[s.sepal_length, s.sepal_width, s.petal_length, s.petal_width]
                  for s in batch.samples])
    X_scaled = models["scaler"].transform(X)
    preds = models["classifier"].predict(X_scaled).tolist()
    probs = models["classifier"].predict_proba(X_scaled).tolist()
    latency = (time.time() - start) * 1000
    return {
        "predictions": [{"class_id": p, "class_name": CLASS_NAMES[p], "probabilities": pr}
                        for p, pr in zip(preds, probs)],
        "batch_size": len(preds),
        "latency_ms": round(latency, 2)
    }

print("ML serving app defined")

ML serving app defined


## Serving Hugging Face Models

In [3]:
# hf_api.py Serving a Hugging Face model

from fastapi import FastAPI
from pydantic import BaseModel
from contextlib import asynccontextmanager
from typing import List
import asyncio

class TextRequest(BaseModel):
    text: str
    max_length: int = 150

class EmbeddingRequest(BaseModel):
    texts: List[str]

pipeline_registry = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Load in executor to avoid blocking event loop
    loop = asyncio.get_event_loop()
    
    # Uncomment for real usage:
    # from transformers import pipeline
    # from sentence_transformers import SentenceTransformer
    # pipeline_registry["sentiment"] = await loop.run_in_executor(
    #     None, lambda: pipeline("sentiment-analysis")
    # )
    # pipeline_registry["embedder"] = await loop.run_in_executor(
    #     None, lambda: SentenceTransformer("all-MiniLM-L6-v2")
    # )
    
    pipeline_registry["sentiment"] = "mock_sentiment_pipeline"
    pipeline_registry["embedder"] = "mock_embedder"
    print("HF models loaded")
    yield
    pipeline_registry.clear()

app = FastAPI(title="HuggingFace Model API", lifespan=lifespan)

@app.post("/sentiment")
async def analyze_sentiment(request: TextRequest):
    loop = asyncio.get_event_loop()
    # Run CPU-bound inference in thread pool to avoid blocking
    # result = await loop.run_in_executor(None, pipeline_registry["sentiment"], request.text)
    # Simulated:
    result = [{"label": "POSITIVE", "score": 0.99}]
    return {"text": request.text, "result": result}

@app.post("/embeddings")
async def get_embeddings(request: EmbeddingRequest):
    loop = asyncio.get_event_loop()
    # embeddings = await loop.run_in_executor(
    #     None, lambda: pipeline_registry["embedder"].encode(request.texts).tolist()
    # )
    embeddings = [[0.1, 0.2, 0.3] for _ in request.texts]  # mock
    return {"embeddings": embeddings, "dimension": len(embeddings[0])}

print("HuggingFace serving app defined")

HuggingFace serving app defined


## Model Versioning & Health Checks

In [4]:
from fastapi import FastAPI
from pydantic import BaseModel
from datetime import datetime

app = FastAPI()

# Model registry with versioning
model_registry = {
    "v1": {"model": None, "loaded_at": None, "accuracy": 0.92},
    "v2": {"model": None, "loaded_at": None, "accuracy": 0.95},
}
active_version = "v2"

@app.get("/health")
async def health_check():
    return {
        "status": "healthy",
        "active_model_version": active_version,
        "available_versions": list(model_registry.keys()),
        "timestamp": datetime.utcnow().isoformat()
    }

@app.get("/models")
async def list_models():
    return {
        "active": active_version,
        "versions": [
            {"version": v, "accuracy": info["accuracy"]}
            for v, info in model_registry.items()
        ]
    }

@app.post("/predict/{version}")
async def predict_versioned(version: str, data: dict):
    if version not in model_registry:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail=f"Model version {version} not found")
    return {"version": version, "prediction": "mock_result"}

## Prometheus Metrics

In [5]:
# pip install prometheus-client

from fastapi import FastAPI, Request
from prometheus_client import Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST
from fastapi.responses import Response
import time

app = FastAPI()

# Define metrics
REQUEST_COUNT = Counter(
    "http_requests_total",
    "Total HTTP requests",
    ["method", "endpoint", "status"]
)
REQUEST_LATENCY = Histogram(
    "http_request_duration_seconds",
    "HTTP request latency",
    ["endpoint"]
)
PREDICTION_COUNT = Counter(
    "ml_predictions_total",
    "Total ML predictions",
    ["model_version", "class"]
)
MODEL_LATENCY = Histogram(
    "ml_inference_duration_seconds",
    "ML model inference latency"
)
ACTIVE_REQUESTS = Gauge(
    "active_requests",
    "Number of active requests"
)

@app.middleware("http")
async def metrics_middleware(request: Request, call_next):
    ACTIVE_REQUESTS.inc()
    start = time.time()
    response = await call_next(request)
    duration = time.time() - start
    REQUEST_COUNT.labels(request.method, request.url.path, response.status_code).inc()
    REQUEST_LATENCY.labels(request.url.path).observe(duration)
    ACTIVE_REQUESTS.dec()
    return response

@app.get("/metrics")
async def metrics():
    return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

@app.post("/predict")
async def predict(data: dict):
    with MODEL_LATENCY.time():
        result = "positive"  # mock prediction
    PREDICTION_COUNT.labels("v2", result).inc()
    return {"prediction": result}

print("Prometheus metrics app defined")

Prometheus metrics app defined


## Dockerfile for FastAPI ML App

In [6]:
dockerfile_content = '''
# Dockerfile for FastAPI ML service
FROM python:3.11-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    gcc \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Create non-root user
RUN useradd -m appuser && chown -R appuser /app
USER appuser

EXPOSE 8000

# Production: gunicorn with uvicorn workers
CMD ["gunicorn", "main:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", \\
     "--bind", "0.0.0.0:8000", "--timeout", "120"]
'''

requirements = '''
fastapi==0.111.0
uvicorn[standard]==0.29.0
gunicorn==22.0.0
scikit-learn==1.5.0
joblib==1.4.2
numpy==1.26.4
pydantic==2.7.1
prometheus-client==0.20.0
python-multipart==0.0.9
'''

with open("Dockerfile.example", "w") as f:
    f.write(dockerfile_content)

with open("requirements_example.txt", "w") as f:
    f.write(requirements)

print("Dockerfile and requirements created")
print("\nBuild: docker build -t ml-api .")
print("Run: docker run -p 8000:8000 ml-api")

Dockerfile and requirements created

Build: docker build -t ml-api .
Run: docker run -p 8000:8000 ml-api


## Additional Learning Resources

### Documentation
- [FastAPI Official Docs](https://fastapi.tiangolo.com/) Complete reference
- [Prometheus Python Client](https://github.com/prometheus/client_python) Metrics
- [Locust Docs](https://docs.locust.io/) Load testing

### Articles & Guides
- [Serving ML Models in Production](https://huyenchip.com/2020/03/09/real-time-machine-learning.html) Chip Huyen
- [ML System Design](https://github.com/chiphuyen/machine-learning-systems-design) Chip Huyen templates
- [FastAPI for ML - Full Guide](https://www.youtube.com/watch?v=1-tJCMm8rX8) YouTube tutorial

### Production Tools
- [BentoML](https://docs.bentoml.com/) ML model serving framework
- [Ray Serve](https://docs.ray.io/en/latest/serve/index.html) Scalable model serving
- [vLLM](https://docs.vllm.ai/) High-throughput LLM serving
- [Triton Inference Server](https://docs.nvidia.com/deeplearning/triton-inference-server/) NVIDIA's server

### Books
- [Building ML Powered Applications](https://www.oreilly.com/library/view/building-machine-learning/9781492045106/) Emmanuel Ameisen